# 01 — Exploração da base Olist

## Objetivo
Conhecer a estrutura e a qualidade inicial da tabela de avaliações antes de preparar a base de conhecimento do RAG.

Este notebook usa a implementação de leitura de `src/data/loader.py`. Os CSVs originais não são versionados no Git: baixe o Brazilian E-Commerce Public Dataset by Olist e coloque os arquivos em `data/raw/`.

**Entrada principal:** `olist_order_reviews_dataset.csv`.

**Saída esperada:** compreensão das dimensões, campos, valores ausentes, distribuição das notas e disponibilidade de comentários.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
from src.data.loader import load_reviews

REVIEWS_FILE = RAW_DIR / "olist_order_reviews_dataset.csv"
reviews = load_reviews(REVIEWS_FILE)

print(f"Linhas: {len(reviews):,}")
print(f"Colunas: {len(reviews.columns)}")
display(reviews.head())


## 1. Estrutura dos dados

Verifique tipos, campos e dimensões para entender o que está disponível na fonte.

In [ ]:
display(reviews.info())
display(reviews.describe(include="all").T)


## 2. Valores ausentes

A ausência de comentário é relevante para o RAG: registros sem texto não podem compor documentos semânticos, mas continuam fazendo parte da tabela original.

In [ ]:
missing = reviews.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(reviews) * 100).round(2)
quality = __import__("pandas").DataFrame({"ausentes": missing, "ausentes_%": missing_pct})
display(quality)


## 3. Distribuição das notas

A nota é um metadado útil para segmentar a voz do cliente, embora não substitua a leitura do comentário.

In [ ]:
import matplotlib.pyplot as plt

score_counts = reviews["review_score"].value_counts().sort_index()
display(score_counts.rename("quantidade").to_frame())
ax = score_counts.plot(kind="bar", rot=0, title="Distribuição das notas das avaliações")
ax.set_xlabel("Nota")
ax.set_ylabel("Quantidade")
plt.show()


## 4. Comentários e datas

Compare o volume de comentários preenchidos e observe a extensão dos textos.

In [ ]:
import pandas as pd

comments = reviews["review_comment_message"].fillna("").astype(str).str.strip()
non_empty = comments.ne("")
lengths = comments[non_empty].str.len()

print(f"Avaliações com comentário: {non_empty.sum():,} ({non_empty.mean():.1%})")
print(f"Avaliações sem comentário: {(~non_empty).sum():,}")
display(lengths.describe().to_frame("caracteres"))
display(reviews["review_creation_date"].head())


## 5. Amostra de avaliações

Inspecione exemplos reais para verificar o idioma, o estilo e o tipo de informação presente nos comentários.

In [ ]:
sample = reviews.loc[non_empty, ["review_id", "review_score", "review_creation_date", "review_comment_message"]].sample(
    n=min(10, int(non_empty.sum())), random_state=42
)
display(sample)


## 6. Conclusões da exploração

Registre aqui, após executar o notebook, observações sobre:

- proporção de avaliações com texto;
- distribuição das notas;
- campos com maior ausência;
- limitações identificadas na fonte;
- decisões que serão tratadas no notebook de preparação.

**Próxima etapa:** [02 — Preparação da base](02_preparacao_base.ipynb).